In [1]:
import csv

data = {}
with open("inputs_baseline.csv", mode="r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        data[row["parameter"].strip()] = float(row["value"])

h2_kg_h = data["h2_output_kg_h"]
grid_kwh_h = data["grid_elec_kwh_h"]
ren_share = data["renewable_elec_share"]
grid_ef = data["grid_ef_kg_co2_per_kwh"]
ren_ef = data["ren_ef_kg_co2_per_kwh"]
heat_mj_h = data["heat_duty_mj_h"]
heat_ef = data["heat_ef_kg_co2_per_mj"]
elec_price = data["elec_price_usd_per_kwh"]

eff_elec_ef = (1.0 - ren_share) * grid_ef + ren_share * ren_ef
elec_ghg_h = grid_kwh_h * eff_elec_ef
heat_ghg_h = heat_mj_h * heat_ef
total_ghg_h = elec_ghg_h + heat_ghg_h

ghg_per_kg_h2 = total_ghg_h / h2_kg_h
elec_cost_per_kg_h2 = (grid_kwh_h * elec_price) / h2_kg_h

print(f"H2 production: {h2_kg_h:.2f} kg/h")
print(f"Effective electricity EF: {eff_elec_ef:.4f} kg CO2e/kWh")
print(f"Total utility GHG: {total_ghg_h:.2f} kg CO2e/h")
print(f"LCA (Utility Carbon Footprint): {ghg_per_kg_h2:.2f} kg CO2e / kg H2")
print(f"TEA (Electricity Utility Cost): ${elec_cost_per_kg_h2:.2f} / kg H2")

H2 production: 30.65 kg/h
Effective electricity EF: 0.3660 kg CO2e/kWh
Total utility GHG: 140.62 kg CO2e/h
LCA (Utility Carbon Footprint): 4.59 kg CO2e / kg H2
TEA (Electricity Utility Cost): $0.16 / kg H2


In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Interactive sliders for key process & energy scenarios
ren_slider = widgets.FloatSlider(value=0.20, min=0.0, max=1.0, step=0.05, description='Renewable %:')
heat_rec_slider = widgets.FloatSlider(value=0.0, min=0.0, max=0.60, step=0.05, description='Heat Recov %:')
elec_use_slider = widgets.FloatSlider(value=50.0, min=20.0, max=200.0, step=10.0, description='Grid kWh/h:')

out = widgets.Output()

def update_twin(change=None):
    with out:
        clear_output(wait=True)
        ren_share = ren_slider.value
        heat_recovery = heat_rec_slider.value
        elec_kwh_h = elec_use_slider.value

        # Effective electricity emission factor
        eff_ef = (1.0 - ren_share) * grid_ef + ren_share * ren_ef

        # Net thermal duty after heat recovery
        net_heat_mj_h = heat_mj_h * (1.0 - heat_recovery)

        # Total GHG and LCA per kg H2
        total_ghg = (elec_kwh_h * eff_ef) + (net_heat_mj_h * heat_ef)
        lca_val = total_ghg / h2_kg_h

        # Total utility OPEX (electricity + thermal fuel equivalent at $0.015/MJ)
        util_cost_h = (elec_kwh_h * elec_price) + (net_heat_mj_h * 0.015)
        tea_util_val = util_cost_h / h2_kg_h

        print("=== CLIMATE DIGITAL TWIN: LIVE SCENARIO ===")
        print(f"Renewable Electricity Share : {ren_share*100:.0f}%")
        print(f"Heat Recovery Efficiency    : {heat_recovery*100:.0f}%")
        print(f"Total Utility GHG           : {total_ghg:.2f} kg CO2e/h")
        print(f"LCA Carbon Footprint        : {lca_val:.2f} kg CO2e / kg H2")
        print(f"TEA Total Utility Cost      : ${tea_util_val:.2f} / kg H2")

ren_slider.observe(update_twin, names='value')
heat_rec_slider.observe(update_twin, names='value')
elec_use_slider.observe(update_twin, names='value')

display(ren_slider, heat_rec_slider, elec_use_slider, out)
update_twin()

FloatSlider(value=0.2, description='Renewable %:', max=1.0, step=0.05)

FloatSlider(value=0.0, description='Heat Recov %:', max=0.6, step=0.05)

FloatSlider(value=50.0, description='Grid kWh/h:', max=200.0, min=20.0, step=10.0)

Output()